# Apache Spark — Crash Course> **Engineering Crash Courses** · [Web verzió](./index.html) · [Vissza a főoldalra](../index.html)Ez egy futtatható **Jupyter notebook** formátum, párhuzamosan a web-alapú kurzussal.Itt ugyanazokat a kódrészleteket tudod lokálisan, saját környezetben végigcsinálni.## Hogyan futtasd```bash# 1. Virtuális környezet (Python 3.10+)python -m venv .venv# Windows:.venv\Scripts\activate# macOS/Linux:source .venv/bin/activate# 2. Telepítsd a függőségeket (a notebook első cellája)# 3. Indítsd a Jupytertjupyter lab# vagyjupyter notebook```Minden cella saját magában értelmezhető. A `# %%` kommentek Jupytekben és VSCode-ban is a cellák határát jelölik.

## 1. Környezet — PySpark lokálisan

In [ ]:
%pip install pyspark==3.5.0 delta-spark==3.0.0 --quiet

In [ ]:
from pyspark.sql import SparkSessionfrom pyspark.sql import functions as Fspark = (    SparkSession.builder      .appName('WebShop-Spark')      .config('spark.sql.shuffle.partitions', '4')      .master('local[*]')      .getOrCreate())spark.sparkContext.setLogLevel('WARN')print('Spark verzió:', spark.version)

## 2. DataFrame létrehozás

In [ ]:
data = [    (1, 'Anna',   'Budapest',  12500, '2025-01-20'),    (2, 'Béla',   'Debrecen',  24000, '2025-02-15'),    (3, 'Csilla', 'Szeged',     5200, '2025-03-01'),    (1, 'Anna',   'Budapest',   8900, '2025-02-15'),    (2, 'Béla',   'Debrecen',  11000, '2025-03-20'),]cols = ['customer_id', 'name', 'city', 'amount', 'order_date']orders = spark.createDataFrame(data, cols)orders.printSchema()orders.show()

## 3. Transzformációk — filter, groupBy, agg

In [ ]:
agg = (orders    .filter(F.col('amount') > 5000)    .groupBy('city')    .agg(        F.count('*').alias('orders'),        F.sum('amount').alias('revenue'),        F.avg('amount').alias('avg_order'),    )    .orderBy(F.desc('revenue')))agg.show()

## 4. Window function — rangsor az ügyfél költéseiben

In [ ]:
from pyspark.sql.window import Windoww = Window.partitionBy('customer_id').orderBy(F.desc('amount'))with_rank = (orders    .withColumn('rank_per_customer', F.row_number().over(w))    .withColumn('running_total', F.sum('amount').over(        Window.partitionBy('customer_id').orderBy('order_date')    )))with_rank.show()

## 5. Spark SQL — ad-hoc lekérdezés

In [ ]:
orders.createOrReplaceTempView('orders')spark.sql('''    SELECT city,           COUNT(*)    AS orders,           SUM(amount) AS revenue    FROM orders    GROUP BY city    ORDER BY revenue DESC''').show()

## 6. Írás Parquet + Delta formátumba

In [ ]:
from pathlib import PathPath('spark-out').mkdir(exist_ok=True)# Parquet(orders.write   .mode('overwrite')   .partitionBy('city')   .parquet('spark-out/orders-parquet'))print('Parquet kiírva')# Bejárjuk a partíciókat!ls -la spark-out/orders-parquet/ 2>/dev/null || dir spark-out\\orders-parquet\\

## 7. Execution plan — miért fontos`explain()` mutatja a Catalyst optimalizáló által generált fizikai tervet.

In [ ]:
agg.explain(mode='formatted')

## 8. SparkSession leállítása

In [ ]:
spark.stop()print('Spark leállítva')

## Következő lépések- Térj vissza a [web-alapú kurzushoz](spark-crash-course/index.html) a teljes anyagért, diagramokért és kvízekért.- Kapcsolódó források és videók a kurzusoldal alján találhatók a "További tanulás" szekcióban.- Ha elakadsz: [GitHub Issues](https://github.com/lugosidomotor/engineering_crash_courses/issues)---*Engineering Crash Courses · MIT licenc · Magyar Data & AI Engineering kurzusok*